# Accountable teams: Colab runner
Runtime > Change runtime type > GPU, then run the cells in order.

In [ ]:
!rm -rf accountable-teams && git clone -q https://github.com/yashwanthsagar2004-dot/accountable-teams.git
%cd accountable-teams
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -e '.[dev]' jumanji
import jax; print(jax.__version__, jax.devices())

In [ ]:
# Build step 1: tests and the tier-1 pilot
!python -m pytest -q
!python -m accteams.train_tier1 --seeds 3 --out experiments/tier1_colab.json

In [ ]:
# Build step 2, first gate run: Level-Based Foraging, window 1 and window 5
for w in (1, 5):
    !python experiments/lbf_smoke.py --decisions 300000 --eval 100 --reps 200 --seeds 3 --train-steps 20000 --batch 2048 --patience 100000 --window {w} --out experiments/lbf_colab_window{w}.json

In [ ]:
# Build step 2, the gate: sweep window x branch length; conditions 1 and 2 need no critic
!git pull -q
!python experiments/snr_gate.py --windows 1,3,5,10 --horizons 5,10,15,25 --eval 60 --reps 400 --out experiments/snr_gate_colab.json

In [ ]:
# Show every result record so it can be copied back
!for f in experiments/*colab*.json; do echo "== $f"; cat $f; done